# 🤖 Notebook 3 — Model Building & Evaluation
**Project:** Customer Churn Prediction  
**Author:** Aditya Bobade

---
### Objectives
- Train Logistic Regression, Random Forest & Gradient Boosting
- Evaluate with Precision, Recall, F1, ROC-AUC
- Compare models visually
- Plot confusion matrices and ROC curves
- Identify top feature importances


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/telco_churn_balanced.csv')
print("Shape:", df.shape)
print("Churn balance:", df['Churn'].value_counts().to_dict())


Shape: (10022, 37)
Churn balance: {0: 5011, 1: 5011}


In [2]:
# Train/Test Split
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


Train: 8,017 | Test: 2,005


In [3]:
# Train all 3 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=150, max_depth=10,
                                                   random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                       max_depth=4, random_state=42)
}

results = {}
for name, model in models.items():
    Xtr = X_train_sc if name=='Logistic Regression' else X_train
    Xte = X_test_sc  if name=='Logistic Regression' else X_test
    model.fit(Xtr, y_train)
    yp   = model.predict(Xte)
    yprb = model.predict_proba(Xte)[:,1]
    results[name] = dict(
        model=model, y_pred=yp, y_prob=yprb,
        precision=precision_score(y_test,yp),
        recall=recall_score(y_test,yp),
        f1=f1_score(y_test,yp),
        roc_auc=roc_auc_score(y_test,yprb),
        Xte=Xte
    )
    print(f"\n{'='*40}\n{name}")
    print(classification_report(y_test, yp, target_names=['Not Churned','Churned']))



Logistic Regression
              precision    recall  f1-score   support

 Not Churned       0.71      0.68      0.69      1003
     Churned       0.69      0.72      0.70      1002

    accuracy                           0.70      2005
   macro avg       0.70      0.70      0.70      2005
weighted avg       0.70      0.70      0.70      2005


Random Forest
              precision    recall  f1-score   support

 Not Churned       0.84      0.70      0.77      1003
     Churned       0.74      0.87      0.80      1002

    accuracy                           0.79      2005
   macro avg       0.79      0.79      0.78      2005
weighted avg       0.79      0.79      0.78      2005


Gradient Boosting
              precision    recall  f1-score   support

 Not Churned       0.76      0.67      0.71      1003
     Churned       0.71      0.79      0.74      1002

    accuracy                           0.73      2005
   macro avg       0.73      0.73      0.73      2005
weighted avg       

In [4]:
# Model comparison bar chart
metrics  = ['precision','recall','f1','roc_auc']
m_labels = ['Precision','Recall','F1 Score','ROC-AUC']
colors   = ['#3498db','#2ecc71','#e74c3c']
x        = np.arange(len(metrics))
width    = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=name,
                  color=colors[i], edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + width); ax.set_xticklabels(m_labels, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../charts/07_model_comparison.png', bbox_inches='tight')
plt.show()


In [5]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Churned','Churned'],
                yticklabels=['Not Churned','Churned'],
                linewidths=1, linecolor='white')
    ax.set_title(f'{name}\nF1:{res["f1"]:.3f} | AUC:{res["roc_auc"]:.3f}',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../charts/08_confusion_matrices.png', bbox_inches='tight')
plt.show()


In [6]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#3498db','#2ecc71','#e74c3c']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={res["roc_auc"]:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../charts/09_roc_curves.png', bbox_inches='tight')
plt.show()


In [7]:
# Feature Importance — Random Forest
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X_train.columns).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(importances.index, importances.values, color='#3498db', edgecolor='white')
for bar, val in zip(bars, importances.values):
    ax.text(val+0.001, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../charts/10_feature_importance.png', bbox_inches='tight')
plt.show()


In [8]:
# 5-Fold Cross Validation on best model (Gradient Boosting)
gb  = results['Gradient Boosting']['model']
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(gb, X_train, y_train, cv=cv, scoring='f1')
print(f"5-Fold CV F1: {cvs.round(3)}")
print(f"Mean: {cvs.mean():.3f}  Std: {cvs.std():.3f}")

print("\n=== Final Comparison ===")
for name, res in results.items():
    print(f"{name:<25} P:{res['precision']:.3f} R:{res['recall']:.3f} F1:{res['f1']:.3f} AUC:{res['roc_auc']:.3f}")


5-Fold CV F1: [0.718 0.747 0.751 0.719 0.749]
Mean: 0.737  Std: 0.015

=== Final Comparison ===
Logistic Regression       P:0.690 R:0.718 F1:0.704 AUC:0.760
Random Forest             P:0.745 R:0.868 F1:0.802 AUC:0.872
Gradient Boosting         P:0.706 R:0.788 F1:0.745 AUC:0.803
